# 👥 Nexora — Customer Intelligence & Behavioral EDA
**Phase 4: Multi-Level EDA | Notebook 02**

### Objective
Deep dive into customer-level behavior, purchase frequencies, retention curves, category diversity, and lifetime ordering trajectories across 206,209 customers.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

DATA_DIR = Path("../data/processed")
df_customers = pd.read_csv(DATA_DIR / "customers.csv")
df_orders = pd.read_csv(DATA_DIR / "orders.csv")
print(f"Loaded {len(df_customers):,} customers and {len(df_orders):,} orders.")


---
## ❓ Business Question 1: What is the lifetime order frequency per customer?
*Helps segment one-time/infrequent buyers from high-frequency brand loyalists.*


In [ ]:
plt.figure(figsize=(12, 5))
sns.histplot(df_customers['total_orders'], bins=50, color='#1d3557', kde=True)
plt.title("Customer Distribution by Total Lifetime Orders", fontsize=14, fontweight='bold')
plt.xlabel("Total Orders per Customer")
plt.ylabel("Number of Customers")
plt.axvline(df_customers['total_orders'].median(), color='red', linestyle='--', label=f"Median Orders: {df_customers['total_orders'].median():.0f}")
plt.legend()
plt.show()


---
## ❓ Business Question 2: Does customer reorder rate increase with order tenure?
*Reveals the customer lifecycle curve: do customers develop predictable grocery habits as they place more orders?*


In [ ]:
# Calculate average reorder rate as order_number increases
df_op_sample = pd.read_csv(DATA_DIR / "order_products.csv", nrows=4000000)
df_merged = df_op_sample.merge(df_orders[['order_id', 'order_number']], on='order_id')

order_progression = df_merged.groupby('order_number')['reordered'].mean().reset_index()

plt.figure(figsize=(12, 5))
sns.lineplot(data=order_progression[order_progression['order_number'] <= 60], x='order_number', y='reordered', color='#2a9d8f', linewidth=2.5)
plt.title("Reorder Rate Progression Across Customer Lifetime Orders", fontsize=14, fontweight='bold')
plt.xlabel("Order Sequence Number")
plt.ylabel("Average Reorder Rate")
plt.ylim(0, 1.0)
plt.show()


---
## ❓ Business Question 3: How consistent are customers in their order intervals?
*Customers with low variance in order interval are habitual shoppers; high-variance shoppers are opportunistic.*


In [ ]:
user_intervals = df_orders[df_orders['is_first_order'] == 0].groupby('user_id')['days_since_prior_order'].agg(
    mean_interval='mean',
    std_interval='std',
    order_count='count'
).dropna()

plt.figure(figsize=(10, 5))
sns.scatterplot(data=user_intervals.sample(5000, random_state=42), x='mean_interval', y='std_interval', alpha=0.3, color='#023047')
plt.title("Customer Purchase Cadence: Mean Interval vs. Interval Standard Deviation", fontsize=13, fontweight='bold')
plt.xlabel("Mean Days Between Orders")
plt.ylabel("Standard Deviation of Order Interval")
plt.show()


---
## ❓ Business Question 4: What is the distribution of department diversity per customer?
*Assessing how many departments a customer orders from indicates full grocery adoption vs. single-niche utility.*


In [ ]:
df_prod = pd.read_csv(DATA_DIR / "products.csv")
df_merged_prod = df_op_sample.merge(df_prod[['product_id', 'department_id']], on='product_id').merge(df_orders[['order_id', 'user_id']], on='order_id')

user_dept_diversity = df_merged_prod.groupby('user_id')['department_id'].nunique()

plt.figure(figsize=(10, 4))
sns.countplot(x=user_dept_diversity, color='#f4a261')
plt.title("Department Diversity Count per Customer (Sample)", fontsize=13, fontweight='bold')
plt.xlabel("Number of Unique Departments Explored")
plt.ylabel("Number of Customers")
plt.show()
